In [2]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [3]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [4]:
print(len(documents))

72


In [5]:
from minsearch import Index

index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(documents)

In [6]:
query = "How does the agentic loop keep calling the model until it stops?"

results = index.search(query, num_results=5)

print(results[0]["filename"])

01-agentic-rag/lessons/14-agentic-loop.md


In [7]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [8]:
def build_context(search_results):
    lines = []
    for doc in search_results:
        lines.append(f"File: {doc['filename']}")
        lines.append(doc['content'])
        lines.append("")
    return "\n".join(lines).strip()

def build_prompt(query, search_results):
    context = build_context(search_results)
    return f"""QUESTION: {query}

CONTEXT:
{context}""".strip()

INSTRUCTIONS = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the course lessons.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

def rag(query):
    results = index.search(query, num_results=5)
    prompt = build_prompt(query, results)
    
    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=[
            {"role": "developer", "content": INSTRUCTIONS},
            {"role": "user", "content": prompt}
        ]
    )
    
    return response.output_text, response.usage

In [9]:
query = "How does the agentic loop keep calling the model until it stops?"
answer, usage = rag(query)
print("Answer:", answer)
print("Input tokens:", usage.input_tokens)

Answer: The loop keeps calling the model by using a `while True` loop and checking whether the latest response included any function calls.

- It sets `has_function_calls = False` before each model call.
- It sends the full `messages` history to the model.
- If the model returns a `function_call`, the code runs the tool, appends the tool result to `messages`, and sets `has_function_calls = True`.
- At the end of the iteration, if `has_function_calls == False`, it breaks out of the loop.

So it keeps looping until the model returns a response with no function calls, which means the model has given its final answer.
Input tokens: 7110


In [10]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)
print(len(chunks))

295


In [11]:
chunk_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

chunk_index.fit(chunks)

In [12]:
def rag_chunked(query):
    results = chunk_index.search(query, num_results=5)
    prompt = build_prompt(query, results)
    
    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=[
            {"role": "developer", "content": INSTRUCTIONS},
            {"role": "user", "content": prompt}
        ]
    )
    
    return response.output_text, response.usage

query = "How does the agentic loop keep calling the model until it stops?"
answer2, usage2 = rag_chunked(query)
print("Answer:", answer2)
print("Input tokens (chunked):", usage2.input_tokens)
print("Input tokens (original):", 7110)
print("Ratio:", round(7110 / usage2.input_tokens, 1), "x fewer")

Answer: The loop keeps calling the model in a `while True` loop and tracks whether any `function_call` items appeared in the model’s output.

- At each turn, it sets `has_function_calls = False`
- It calls the model
- If the output includes a `function_call`, it runs the tool, appends the result, and sets `has_function_calls = True`
- If there are no function calls on that turn, it breaks out of the loop

So the stop condition is: **no function calls this turn means the model is done, and the loop ends.**
Input tokens (chunked): 2293
Input tokens (original): 7110
Ratio: 3.1 x fewer


In [13]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

def search(query: str) -> list:
    """
    Search the course lessons for entries matching the given query.
    """
    return chunk_index.search(
        query,
        num_results=5
    )

agent_tools = Tools()
agent_tools.add_tool(search)

In [14]:
agent_instructions = """
You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering.
""".strip()

chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=agent_instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [15]:
result = runner.loop(
    prompt="How does the agentic loop work, and how is it different from plain RAG?",
    callback=callback,
)

-> Response received


-> Response received
